# Naive offline CRL on the litter AntMaze dataset (Colab GPU)

Trains the **faithful offline contrastive-RL recipe** (bc 0.05, twin-min, alpha 0,
batch 1024, repr 16, hidden (1024,1024)) on the frozen litter dataset
`antmaze_litter_full.npz` (obs/act only -- no sidecar, no U, no privileged labels).
Eval runs in `offline_ant_umaze_litter` (collapse ON) so lane/collapse are visible.

Same structure as `offline_d4rl_antmaze_50k.ipynb`. GPU accelerates the matmul-bound
training; the litter-env eval rollouts stay CPU/MuJoCo-bound (a small fraction of time).
Behavioural characterisation (lane/speed/per-U) is best done LOCALLY on the downloaded
checkpoint (simulator-bound) via `scripts/eval_naive_litter.py`.

## 1. Configuration -- single source of truth

In [ ]:
# 1. Configuration. Everything tunable lives here.
import os

SEED        = 0
MAX_UPDATES = 60_000          # learner updates (diagnostic). Raise for the causal method.
RESUME      = False           # set True after a Colab disconnect (restores latest.pkl)
REQUIRE_GPU = True

# --- repo (pin a COMMIT for reproducibility; '' = tip of BRANCH) --------------
REPO     = 'contrastive_rl'
REPO_URL = 'https://github.com/tingrui-huang/contrastive_rl.git'
BRANCH   = 'main'
COMMIT   = ''                 # e.g. 'bb14fff'; '' -> origin/main tip

# --- dataset (a FIXED npz; NOT produced by this run) --------------------------
DATASET_DRIVE_PATH = '/content/drive/MyDrive/contrastive_rl_datasets/litter/antmaze_litter_full.npz'
LOCAL_DATASET_PATH = '/content/antmaze_litter_full.npz'
DATASET_SHA_PREFIX = '715aaeb832283f2b'   # frozen full dataset identity

# --- run / output dirs --------------------------------------------------------
RUN_ID        = 'naive_litter_crl_s0_60k'
RUN_DRIVE_DIR = f'/content/drive/MyDrive/contrastive_rl_runs/litter/{RUN_ID}'
LOCAL_RUN_DIR = f'/content/scratch_{RUN_ID}'     # fast local scratch (atomic writes)

ENV_NAME = 'offline_ant_umaze_litter'
print('RUN_ID', RUN_ID, '| env', ENV_NAME, '| updates', MAX_UPDATES)

## 2. Mount Google Drive; create dirs

In [ ]:
# 2. Mount Drive; make local scratch + Drive run trees.
from google.colab import drive
drive.mount('/content/drive')
for base in (LOCAL_RUN_DIR, RUN_DRIVE_DIR):
    os.makedirs(base, exist_ok=True)
os.makedirs(os.path.dirname(DATASET_DRIVE_PATH), exist_ok=True)
print('local scratch:', LOCAL_RUN_DIR)
print('Drive run dir:', RUN_DRIVE_DIR)
!df -h /content | tail -1

## 3. Clone/checkout the repo (refuses to overwrite uncommitted work)

In [ ]:
# 3. Clone if absent; otherwise fetch + checkout at COMMIT (or origin/BRANCH).
import subprocess, sys
os.chdir('/content')
if not os.path.exists(REPO):
    !git clone $REPO_URL $REPO
os.chdir(REPO)
dirty = subprocess.run(['git','status','--porcelain'], capture_output=True, text=True).stdout.strip()
if dirty:
    raise RuntimeError('repo has uncommitted changes -- refusing to checkout over them:\n' + dirty)
!git fetch -q origin
ref = COMMIT if COMMIT else f'origin/{BRANCH}'
!git checkout -q $ref
!git log -1 --oneline
for req in ('crl/offline_audit.py','crl/d4rl_ant.py','crl/train.py',
            'scripts/naive_litter_crl.py','scripts/verify_offline_d4rl.py',
            'scripts/eval_naive_litter.py'):
    if not os.path.exists(req):
        raise RuntimeError(f'{req} missing -- push main from the workstation and rerun.')
print('litter pipeline files present -- checkout OK')

## 4. Dependencies (preserve Colab's preinstalled GPU JAX -- do not alter)

In [ ]:
# 4. Install deps WITHOUT disturbing Colab's GPU JAX (pin jax/jaxlib/numpy).
os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'
os.environ['MUJOCO_GL'] = 'egl'
import jax, jaxlib, numpy
hold = [f'jax=={jax.__version__}', f'jaxlib=={jaxlib.__version__}', f'numpy=={numpy.__version__}']
def pip(*a): subprocess.run([sys.executable,'-m','pip','install','-q',*a], check=True)
print('Colab JAX', jax.__version__, '| devices:', jax.devices())
pip('--no-deps', 'dm-haiku', 'optax', 'chex')
pip('jmp', 'tabulate', 'toolz', 'etils', 'tensorboardX', 'mujoco', 'imageio', 'imageio-ffmpeg', *hold)
print('post-install JAX', jax.__version__, '| devices:', jax.devices())

## 5. GPU / environment verification

In [ ]:
# 5. Require an accelerator; record env meta to Drive.
import hashlib, json, platform, mujoco
os.chdir('/content/'+REPO)
commit = subprocess.run(['git','rev-parse','HEAD'], capture_output=True, text=True).stdout.strip()
meta = {'run_id': RUN_ID, 'python': platform.python_version(), 'jax': jax.__version__,
        'backend': jax.default_backend(), 'devices':[str(d) for d in jax.devices()],
        'mujoco': mujoco.__version__, 'git_commit': commit, 'env': ENV_NAME}
if REQUIRE_GPU and jax.default_backend() == 'cpu':
    raise RuntimeError('no accelerator -- Runtime > Change runtime type > GPU')
!nvidia-smi -L || true
json.dump(meta, open(f'{RUN_DRIVE_DIR}/meta_env.json','w'), indent=2)
print(json.dumps(meta, indent=1))

## 6. Dataset: verify the Drive npz -> copy to /content -> re-verify SHA

Upload `artifacts/litter_dataset/full/antmaze_litter_full.npz` to the Drive path in section 1 first.

In [ ]:
# 6. Fetch the fixed litter npz from Drive; fail fast if absent or wrong file.
import shutil
def sha256(path, chunk=1<<20):
    h = hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda: f.read(chunk), b''): h.update(b)
    return h.hexdigest()
if not os.path.exists(DATASET_DRIVE_PATH):
    raise FileNotFoundError(
        f'\n=== DATASET MISSING ON DRIVE ===\nexpected: {DATASET_DRIVE_PATH}\n'
        'Upload antmaze_litter_full.npz (workstation artifacts/litter_dataset/full/)\n'
        'to that Drive path, then rerun. This notebook NEVER collects data.')
dsha = sha256(DATASET_DRIVE_PATH)
print(f'Drive npz: {os.path.getsize(DATASET_DRIVE_PATH)/1e6:.1f} MB  sha256={dsha[:16]}')
if not dsha.startswith(DATASET_SHA_PREFIX):
    raise RuntimeError(f'dataset sha mismatch: expected prefix {DATASET_SHA_PREFIX}, got {dsha[:16]}')
shutil.copy2(DATASET_DRIVE_PATH, LOCAL_DATASET_PATH + '.tmp')
os.replace(LOCAL_DATASET_PATH + '.tmp', LOCAL_DATASET_PATH)
assert sha256(LOCAL_DATASET_PATH) == dsha, 'copied npz sha differs -- bad copy'
print('local copy verified:', LOCAL_DATASET_PATH)

## 7. Pre-training offline audit -- must PASS before any training

In [ ]:
# 7. Static offline gates (G1-G8) on the REAL litter dataset.
os.chdir('/content/'+REPO); sys.path.insert(0, '/content/'+REPO)
from crl import offline_audit
from crl.config import Config
from crl import envs as envs_mod
_c = Config(env_name=ENV_NAME, offline_dataset=LOCAL_DATASET_PATH)
envs_mod.make_env(ENV_NAME, _c, seed=0)   # fills obs/goal/action dims
passed, gates, rep = offline_audit.run_static_audit(LOCAL_DATASET_PATH, _c)
print('offline_audit gates:', {g:('PASS' if ok else 'FAIL') for g,ok in gates.items()})
fp = rep['fingerprint']
print(f"  sha256={fp['sha256'][:16]}  eps={fp['n_episodes']}  trans={fp['n_transitions']}  obs={fp['obs_shape']}")
assert passed, 'offline_audit FAILED -- see gates'

## 8. (resume) restore latest.pkl from Drive

In [ ]:
# 8. If RESUME, pull the last checkpoint from Drive into local scratch.
import glob
if RESUME:
    for f in glob.glob(f'{RUN_DRIVE_DIR}/*.pkl') + [f'{RUN_DRIVE_DIR}/metrics.json',
                                                    f'{RUN_DRIVE_DIR}/offline_dataset.sha256']:
        if os.path.exists(f): shutil.copy2(f, f'{LOCAL_RUN_DIR}/{os.path.basename(f)}')
    print('restored:', sorted(os.path.basename(p) for p in glob.glob(f'{LOCAL_RUN_DIR}/*.pkl')))
else:
    print('fresh run (RESUME=False)')

## 9. TensorBoard

In [ ]:
tb = f'{LOCAL_RUN_DIR}/tb'
%load_ext tensorboard
%tensorboard --logdir $tb

## 10. Launch training (live stream)

Trains to Drive-mirrored local scratch. Eval every 10k in the litter env (collapse ON).

In [ ]:
# 10. Launch naive offline CRL on the litter dataset (LIVE streaming).
os.chdir('/content/'+REPO)
os.environ['PYTHONUNBUFFERED'] = '1'
cmd = [sys.executable, '-u', 'scripts/naive_litter_crl.py',
       '--npz', LOCAL_DATASET_PATH, '--steps', str(MAX_UPDATES),
       '--seed', str(SEED), '--ckpt-dir', LOCAL_RUN_DIR]
if RESUME: cmd.append('--resume')
print(' '.join(cmd)); print('-'*70)
proc = subprocess.Popen(cmd, env={**os.environ}, stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='')
rc = proc.wait()
print(f'\n[driver exited {rc}]')

## 11. Mirror checkpoints to Drive

In [ ]:
# 11. Copy checkpoints + metrics to the persistent Drive dir.
import glob
n = 0
for f in glob.glob(f'{LOCAL_RUN_DIR}/*.pkl') + glob.glob(f'{LOCAL_RUN_DIR}/*.json') + glob.glob(f'{LOCAL_RUN_DIR}/*.sha256'):
    shutil.copy2(f, f'{RUN_DRIVE_DIR}/{os.path.basename(f)}'); n += 1
print(f'mirrored {n} files to {RUN_DRIVE_DIR}')
mp = f'{LOCAL_RUN_DIR}/metrics.json'
if os.path.exists(mp):
    for e in json.load(open(mp))[-8:]:
        print({k: e.get(k) for k in ('step','success','min_dist','final_dist')})

## 12. Behavioural characterisation of best.pkl (litter env, collapse ON)

MuJoCo-bound; 100 eps here is fine, or download `best.pkl` and run locally:
`python scripts/eval_naive_litter.py --ckpt best.pkl --eps 200`. Reports success,
per-U success, collapse/fall/timeout, corridor lane and speed, and the
characterization (fixed-side / fast-middle / middle-slow / unstable mixture).

In [ ]:
# 12. Behavioural eval of the best checkpoint (100 eps here for speed).
os.chdir('/content/'+REPO)
r = subprocess.run([sys.executable, 'scripts/eval_naive_litter.py',
                    '--ckpt', f'{LOCAL_RUN_DIR}/best.pkl', '--eps', '100',
                    '--out', f'{RUN_DRIVE_DIR}/behaviour_eval.json'],
                   capture_output=True, text=True)
print(r.stdout[-2000:]); print(r.stderr[-800:] if r.returncode else '')